# Notebook 1: primary high-order models

Fits the selected BGNAR-SV specification, its matched variance ablation, the nested cross-sectional ladder and matched observed-topology sensitivities.

In [1]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import time
import numpy as np
import pandas as pd
import pymc as pm

QUICK = quick_mode()
selection = load_result("nb0_selection")["selection"]
P = int(selection["p"])
K = int(selection["k"])
assert P == 38 and selection["network"] == "geographic" and K == 2
assert int(selection["max_stage"]) == 2

d = load_data(network="geographic")
Y_train = d["Y_train"].to_numpy()
Y_full = d["Y_full"].to_numpy()
test_start = d["test_start"]
Y_hist = Y_full[:test_start]
Y_future_full = Y_full[test_start:]
test_dates_full = pd.to_datetime(d["test"].index)
N = d["N"]
networks = d["networks"]

EVAL_STEPS = min(3, len(Y_future_full)) if QUICK else len(Y_future_full)
Y_future = Y_future_full[:EVAL_STEPS]
test_dates = test_dates_full[:EVAL_STEPS]

W_geo = knn_sparsify(networks["geographic"], K)
W_export = knn_sparsify(networks["export"], K)
W_import = knn_sparsify(networks["import"], K)
W_uniform = (np.ones((N, N)) - np.eye(N)) / (N - 1)

S2 = [2] * P
S1 = [1] * P
S0 = [0] * P

for name, Wm in [("geographic", W_geo), ("export", W_export), ("import", W_import)]:
    sw = compute_stage_weights(Wm, 2)
    assert int((sw[1] > 0).sum()) > 0, f"{name} k=2 has no exact stage-2 neighbours"

kw = dict(NUTS_KW)
if QUICK:
    kw.update(draws=2, tune=2, chains=1, cores=1)

def fit_check(model, label):
    """Sample one model and return timing and convergence diagnostics."""
    with model:
        t0 = time.time()
        idata = pm.sample(**kw)
    elapsed = time.time() - t0
    diag = mcmc_diagnostics(idata)
    print(
        f"{label}: {elapsed:.1f}s, div={diag['divergences']}, "
        f"max R-hat={diag['max_rhat']:.5g}, min bulk ESS={diag['min_ess_bulk']:.5g}, "
        f"min BFMI={diag['min_bfmi']:.5g}"
    )
    return idata, elapsed, diag

def forecast_config(model, network, stages, k=None):
    return run_config(
        p=P, stages=stages, model=model, network=network, k=k,
        forecast_horizon=1, seed=NUTS_KW["random_seed"],
    )

## Model fits

In [2]:
specs = {
    "selected_sv": dict(W=W_geo, stages=S2, mode="global_gnar", variance="sv",
                        label="geographic k=2 stage-2 BGNAR-SV"),
    "selected_constant": dict(W=W_geo, stages=S2, mode="global_gnar", variance="constant",
                              label="geographic k=2 stage-2 constant GNAR"),
    "geo_stage1_sv": dict(W=W_geo, stages=S1, mode="global_gnar", variance="sv",
                          label="geographic k=2 stage-1 BGNAR-SV"),
    "ar_sv": dict(W=W_geo, stages=S0, mode="ar_only", variance="sv",
                   label="shared AR(38)-SV"),
    "uniform_stage1_sv": dict(W=W_uniform, stages=S1, mode="global_gnar", variance="sv",
                              label="uniform stage-1 BGNAR-SV"),
    "export_stage2_sv": dict(W=W_export, stages=S2, mode="global_gnar", variance="sv",
                             label="export k=2 stage-2 BGNAR-SV"),
    "import_stage2_sv": dict(W=W_import, stages=S2, mode="global_gnar", variance="sv",
                             label="import k=2 stage-2 BGNAR-SV"),
}

idatas, runtimes, diagnostics, designs = {}, {}, {}, {}

def fit_spec(key):
    """Fit one configured high-order model."""
    spec = specs[key]
    X, y, names = build_design(
        Y_train, spec["W"], p=P, stages=spec["stages"], mode=spec["mode"], h=1
    )
    tidx = build_time_index(Y_train, p=P, h=1)
    assert X.shape[0] == len(tidx) * N

    if spec["variance"] == "sv":
        model = build_gnar_sv(X, y, tidx, names, N)
    else:
        model = build_gnar_constant(X, y, names, N)

    idata, elapsed, diag = fit_check(model, spec["label"])
    idatas[key] = idata
    runtimes[key] = elapsed
    diagnostics[key] = diag
    designs[key] = names

In [3]:
fit_spec("selected_sv")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

In [4]:
fit_spec("selected_constant")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, sigma]


Output()

In [5]:
fit_spec("geo_stage1_sv")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 394 seconds.


geographic k=2 stage-1 BGNAR-SV: 397.2s, div=0, max R-hat=1, min bulk ESS=970, min BFMI=0.74427


In [6]:
fit_spec("ar_sv")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 238 seconds.


shared AR(38)-SV: 241.0s, div=0, max R-hat=1, min bulk ESS=1200, min BFMI=0.70063


In [7]:
fit_spec("uniform_stage1_sv")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 2_000 tune and 1_000 draw iterations (8_000 + 4_000 draws total) took 391 seconds.


uniform stage-1 BGNAR-SV: 393.9s, div=0, max R-hat=1, min bulk ESS=1200, min BFMI=0.67315


In [8]:
fit_spec("export_stage2_sv")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

In [9]:
fit_spec("import_stage2_sv")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, m_h, phi, sigma_h, h_innov]


Output()

In [10]:
assert len(designs["selected_sv"]) == 3 * P
assert len(designs["selected_constant"]) == 3 * P
assert len(designs["geo_stage1_sv"]) == 2 * P
assert len(designs["uniform_stage1_sv"]) == 2 * P
assert len(designs["ar_sv"]) == P
assert len(designs["export_stage2_sv"]) == 3 * P
assert len(designs["import_stage2_sv"]) == 3 * P

## Forecast evaluation

In [11]:
FORECAST_NAMES = {
    "selected_sv": "nb1_geo_k2_s2_sv_p38",
    "selected_constant": "nb1_geo_k2_s2_constant_p38",
    "geo_stage1_sv": "nb1_geo_k2_s1_sv_p38",
    "ar_sv": "nb1_ar_sv_p38",
    "uniform_stage1_sv": "nb1_uniform_s1_sv_p38",
    "export_stage2_sv": "nb1_export_k2_s2_sv_p38",
    "import_stage2_sv": "nb1_import_k2_s2_sv_p38",
}

forecasts, scores, losses = {}, {}, {}
for key, spec in specs.items():
    idata = idatas[key]
    if spec["variance"] == "constant":
        mean, var = forecast_constant(
            idata, Y_hist, Y_future, spec["W"],
            p=P, stages=spec["stages"], mode=spec["mode"],
        )
    else:
        mean, var = forecast_online(
            idata, Y_hist, Y_future, spec["W"],
            p=P, stages=spec["stages"], mode=spec["mode"],
        )

    assert mean.shape == var.shape == Y_future.shape
    score = score_forecasts(Y_future, mean, var, test_dates)
    loss = crps_series(Y_future, mean, var)
    forecasts[key] = (mean, var)
    scores[key] = score
    losses[key] = loss

    network = (
        None if key == "ar_sv"
        else "geographic" if key in {"selected_sv", "selected_constant", "geo_stage1_sv"}
        else "uniform" if key == "uniform_stage1_sv"
        else "export" if key == "export_stage2_sv" else "import"
    )
    k_value = K if network in {"geographic", "export", "import"} else None
    save_forecasts(
        FORECAST_NAMES[key], Y_future, mean, var, test_dates,
        config=forecast_config(spec["label"], network, spec["stages"], k_value),
    )

score_table = pd.DataFrame([
    {"model": key, **{m: scores[key][m] for m in ["CRPS", "RMSE", "MAE", "Coverage_95", "Width"]}}
    for key in specs
])
print(score_table.round(6).to_string(index=False))

paired = {
    "selected_sv_minus_constant": losses["selected_sv"] - losses["selected_constant"],
    "geo_stage2_minus_stage1": losses["selected_sv"] - losses["geo_stage1_sv"],
    "geo_stage1_minus_uniform": losses["geo_stage1_sv"] - losses["uniform_stage1_sv"],
    "uniform_minus_ar": losses["uniform_stage1_sv"] - losses["ar_sv"],
    "geo_stage2_minus_export": losses["selected_sv"] - losses["export_stage2_sv"],
    "geo_stage2_minus_import": losses["selected_sv"] - losses["import_stage2_sv"],
}
for label, diff in paired.items():
    print(f"{label}: mean monthly CRPS difference={diff.mean():+.6f}")

            model     CRPS     RMSE      MAE  Coverage_95    Width
      selected_sv 0.240569 0.472838 0.324720     0.904518 1.384965
selected_constant 0.243041 0.470309 0.323717     0.884058 1.320733
    geo_stage1_sv 0.240988 0.473675 0.325577     0.905371 1.385470
            ar_sv 0.245975 0.483076 0.332292     0.906223 1.406472
uniform_stage1_sv 0.240032 0.472896 0.323622     0.907928 1.386411
 export_stage2_sv 0.240735 0.473739 0.323871     0.905797 1.379524
 import_stage2_sv 0.240402 0.473214 0.323796     0.905797 1.386819
selected_sv_minus_constant: mean monthly CRPS difference=-0.002472
geo_stage2_minus_stage1: mean monthly CRPS difference=-0.000419
geo_stage1_minus_uniform: mean monthly CRPS difference=+0.000956
uniform_minus_ar: mean monthly CRPS difference=-0.005943
geo_stage2_minus_export: mean monthly CRPS difference=-0.000165
geo_stage2_minus_import: mean monthly CRPS difference=+0.000167


## Posterior coefficient summaries

In [12]:
def posterior_term_summary(idata, names):
    """Summarise posterior draws for labelled mean coefficients."""
    draws = np.asarray(idata.posterior["beta"].values, dtype=float).reshape(-1, len(names))
    rows = []
    for j, name in enumerate(names):
        x = draws[:, j]
        rows.append({
            "term": name,
            "mean": float(x.mean()),
            "sd": float(x.std()),
            "lo95": float(np.percentile(x, 2.5)),
            "hi95": float(np.percentile(x, 97.5)),
            "p_gt0": float(np.mean(x > 0)),
        })
    return pd.DataFrame(rows), draws

coef_selected, beta_selected = posterior_term_summary(
    idatas["selected_sv"], designs["selected_sv"]
)

def sum_profile(draws, names, prefix):
    idx = [i for i, name in enumerate(names) if name.startswith(prefix)]
    x = draws[:, idx].sum(axis=1)
    return {
        "mean": float(x.mean()),
        "sd": float(x.std()),
        "lo95": float(np.percentile(x, 2.5)),
        "hi95": float(np.percentile(x, 97.5)),
    }

cumulative = {
    "A_cum": sum_profile(beta_selected, designs["selected_sv"], "own_"),
}
idx_b1 = [i for i, n in enumerate(designs["selected_sv"]) if n.endswith("_stage1")]
idx_b2 = [i for i, n in enumerate(designs["selected_sv"]) if n.endswith("_stage2")]
for key, idx in [("B1_cum", idx_b1), ("B2_cum", idx_b2)]:
    x = beta_selected[:, idx].sum(axis=1)
    cumulative[key] = {
        "mean": float(x.mean()), "sd": float(x.std()),
        "lo95": float(np.percentile(x, 2.5)),
        "hi95": float(np.percentile(x, 97.5)),
    }

Psv = idatas["selected_sv"].posterior
volatility = {}
for name in ["phi", "m_h", "sigma_h"]:
    x = np.asarray(Psv[name].values, dtype=float).ravel()
    volatility[name] = {
        "mean": float(x.mean()), "sd": float(x.std()),
        "lo95": float(np.percentile(x, 2.5)),
        "hi95": float(np.percentile(x, 97.5)),
    }

print(coef_selected.round(6).to_string(index=False))
print("\nCumulative lag summaries:")
print(pd.DataFrame(cumulative).T.round(6).to_string())

            term      mean       sd      lo95      hi95   p_gt0
        own_lag1  1.112438 0.012738  1.087289  1.137805 1.00000
        own_lag2 -0.122041 0.017894 -0.157867 -0.087241 0.00000
        own_lag3  0.037094 0.016899  0.003569  0.069411 0.98450
        own_lag4  0.016302 0.016461 -0.015390  0.048521 0.83950
        own_lag5 -0.024817 0.015331 -0.053881  0.005956 0.05675
        own_lag6  0.029032 0.014689  0.000437  0.057871 0.97625
        own_lag7 -0.032571 0.014204 -0.059452 -0.004578 0.01150
        own_lag8 -0.014204 0.013169 -0.039705  0.011923 0.13725
        own_lag9  0.004053 0.012619 -0.019687  0.028845 0.62300
       own_lag10 -0.009165 0.012075 -0.033339  0.014543 0.22250
       own_lag11 -0.061993 0.011546 -0.084641 -0.039256 0.00000
       own_lag12 -0.175507 0.011592 -0.198478 -0.152784 0.00000
       own_lag13  0.173023 0.011100  0.151616  0.194460 1.00000
       own_lag14  0.058333 0.010172  0.037932  0.078149 1.00000
       own_lag15  0.008971 0.009683 -0.0

## Posterior stability

In [13]:
STABILITY_MODELS = {
    "selected_sv": (idatas["selected_sv"], designs["selected_sv"], W_geo),
    "selected_constant": (idatas["selected_constant"], designs["selected_constant"], W_geo),
    "geo_stage1_sv": (idatas["geo_stage1_sv"], designs["geo_stage1_sv"], W_geo),
    "ar_sv": (idatas["ar_sv"], designs["ar_sv"], W_geo),
    "uniform_stage1_sv": (idatas["uniform_stage1_sv"], designs["uniform_stage1_sv"], W_uniform),
    "export_stage2_sv": (idatas["export_stage2_sv"], designs["export_stage2_sv"], W_export),
    "import_stage2_sv": (idatas["import_stage2_sv"], designs["import_stage2_sv"], W_import),
}

assert P * N == 874

# Cross-check the high-order sparse eigensolver against the explicit companion matrix.
beta_check = np.asarray(
    idatas["selected_sv"].posterior["beta"].values, dtype=float
)[0, 0]
lag_matrices_check = gnar_lag_matrices(
    beta_check, designs["selected_sv"], W_geo
)
rho_sparse_check = companion_radius(lag_matrices_check, method="sparse")
rho_dense_check = companion_radius(lag_matrices_check, method="dense")

np.testing.assert_allclose(
    rho_sparse_check, rho_dense_check, rtol=1e-8, atol=1e-10
)
print(
    f"stability cross-check: sparse={rho_sparse_check:.10f}, "
    f"dense={rho_dense_check:.10f}"
)

stability_radii, stability_summary = {}, {}

for key, (idata, names, Wm) in STABILITY_MODELS.items():
    t0 = time.time()
    beta_draws = np.asarray(idata.posterior["beta"].values, dtype=float)
    radii = posterior_gnar_radii(beta_draws, names, Wm)
    stability_radii[key] = radii
    stability_summary[key] = {
        "median": float(np.median(radii)),
        "q95": float(np.percentile(radii, 95)),
        "max": float(np.max(radii)),
        "stable_fraction": float(np.mean(radii < 1)),
        "unstable_fraction": float(np.mean(radii >= 1)),
        "draws": int(len(radii)),
        "runtime_sec": float(time.time() - t0),
    }
    print(key, stability_summary[key])

stability cross-check: sparse=0.9899404698, dense=0.9899404698


selected_sv {'median': 0.9889360078666194, 'q95': 0.9932628245436752, 'max': 0.9986356157177373, 'stable_fraction': 1.0, 'unstable_fraction': 0.0, 'draws': 4000, 'runtime_sec': 496.10818099975586}


selected_constant {'median': 0.9901693064541621, 'q95': 0.994302705706171, 'max': 0.998807805111805, 'stable_fraction': 1.0, 'unstable_fraction': 0.0, 'draws': 4000, 'runtime_sec': 488.5110137462616}


geo_stage1_sv {'median': 0.9881261462437096, 'q95': 0.9925828388111713, 'max': 0.9970362499417805, 'stable_fraction': 1.0, 'unstable_fraction': 0.0, 'draws': 4000, 'runtime_sec': 45.15728259086609}
ar_sv {'median': 0.9862341363961047, 'q95': 0.9900303782863678, 'max': 0.9943661309397316, 'stable_fraction': 1.0, 'unstable_fraction': 0.0, 'draws': 4000, 'runtime_sec': 0.6687109470367432}


uniform_stage1_sv {'median': 0.9877358178841882, 'q95': 0.9941971997753477, 'max': 1.0025149860422922, 'stable_fraction': 0.99725, 'unstable_fraction': 0.00275, 'draws': 4000, 'runtime_sec': 44.83236765861511}


export_stage2_sv {'median': 0.9987031232840758, 'q95': 1.0106170123889717, 'max': 1.0239907709556009, 'stable_fraction': 0.581, 'unstable_fraction': 0.419, 'draws': 4000, 'runtime_sec': 195.08288407325745}


import_stage2_sv {'median': 1.0109354549755287, 'q95': 1.0280612631167518, 'max': 1.0530000843607177, 'stable_fraction': 0.17575, 'unstable_fraction': 0.82425, 'draws': 4000, 'runtime_sec': 262.73567509651184}


## Save results

In [14]:
save_result("nb1_primary_models", {
    "selection": selection,
    "scores": scores,
    "runtime_sec": runtimes,
    "diagnostics": diagnostics,
    "mean_parameter_counts": {k: len(v) for k, v in designs.items()},
    "paired_monthly_losses": {k: v.tolist() for k, v in paired.items()},
    "monthly_crps": {k: v.tolist() for k, v in losses.items()},
    "coefficient_summary_selected": coef_selected.to_dict(orient="records"),
    "cumulative_lag_summaries": cumulative,
    "volatility": volatility,
    "stability": stability_summary,
    "stability_radii": {k: v.tolist() for k, v in stability_radii.items()},
    "forecast_bundles": FORECAST_NAMES,
    "config": run_config(
        p=P, stages=S2, network="geographic", k=K, max_stage=2,
        purpose="primary_observed_panel",
    ),
    "quick": QUICK,
})
print("saved nb1_primary_models")

saved nb1_primary_models
